In [1]:
# -*- coding: utf-8 -*-
"""
MedGemma CXR — Proper Multi-Task Notebook (FIXED)

This version is consistent with current transformers behavior.
"""

# =========================
# Environment setup
# =========================
!pip install -q transformers accelerate torch torchvision pillow scikit-learn faiss-cpu kagglehub

# =========================
# Imports & device
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
import os

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText
)

from torchvision import transforms
import faiss
import kagglehub

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Load MedGemma (CHAT MODEL, FROZEN)
# =========================
MODEL_ID = "google/medgemma-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

# load the multimodal model that supports .generate()
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"   # avoid double-loading large weights on one GPU
)

model.eval()
# optionally ensure no gradients (no .to(device) when device_map used)
for p in model.parameters():
    p.requires_grad = False

# =========================
# Embedding extractor (SUPPORTED PATH)
# =========================
def extract_embedding(image: Image.Image):
    """
    Extract a pooled embedding from MedGemma hidden states
    using the officially supported chat-template path.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Analyze this image."}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Last hidden state → pooled embedding
    hidden = outputs.hidden_states[-1]   # [1, seq_len, hidden_dim]
    embedding = hidden.mean(dim=1)
    embedding = torch.nan_to_num(embedding, nan=0.0, posinf=0.0, neginf=0.0)
    embedding = F.normalize(embedding, dim=-1, eps=1e-6)


    return embedding  # [1, D]

# =========================
# Load test Chest X-ray (Kaggle)
# =========================
path = kagglehub.dataset_download("assemelqirsh/chest-x-ray-dataset")
print("Dataset path:", path)

img_path = (
    "/root/.cache/kagglehub/datasets/assemelqirsh/"
    "chest-x-ray-dataset/versions/1/"
    "chest_xray/test/PNEUMONIA/person1002_bacteria_2933.jpeg"
)

img = Image.open(img_path)
img

# =========================
# Infer embedding dimension (CORRECT)
# =========================
embedding = extract_embedding(img).float()
EMB_DIM = embedding.shape[-1]
print("Embedding dim:", EMB_DIM)

# =========================
# Classifier head
# =========================
class CXRClassifier(nn.Module):
    def __init__(self, dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_labels)
        )

    def forward(self, x):
        return self.net(x)

NUM_LABELS = 14
classifier = CXRClassifier(EMB_DIM, NUM_LABELS).to(device)
classifier.eval()  # random weights for demo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.6 MB/s eta 0:00:00
Device: cuda


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

100%|██████████| 2.41G/2.41G [00:13<00:00, 188MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/assemelqirsh/chest-x-ray-dataset/versions/1
Embedding dim: 2560


CXRClassifier(
  (net): Sequential(
    (0): Linear(in_features=2560, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=14, bias=True)
  )
)

In [2]:
print(type(model))
print("has generate:", hasattr(model, "generate"))

<class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration'>
has generate: True


In [3]:
# =========================
# Inference (ONLY decision path)
# =========================
LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia"
]

with torch.no_grad():
    logits = classifier(embedding)
    probs = torch.sigmoid(logits).cpu().numpy()[0]

print("=== Risk-scored outputs ===")
for lbl, p in zip(LABELS, probs):
    print(f"{lbl:20s}: {p:.3f}")

# =========================
# Optional retrieval head (FAISS)
# =========================
index = faiss.IndexFlatIP(EMB_DIM)
index.add(embedding.cpu().numpy())

D, I = index.search(embedding.cpu().numpy(), k=1)
print("Retrieved case index:", I[0][0])

# =========================
# Load MedGemma decoder (TEXT ONLY)
# =========================

# decoder.eval()



=== Risk-scored outputs ===
Atelectasis         : 0.502
Cardiomegaly        : 0.506
Effusion            : 0.493
Infiltration        : 0.491
Mass                : 0.509
Nodule              : 0.491
Pneumonia           : 0.499
Pneumothorax        : 0.508
Consolidation       : 0.509
Edema               : 0.505
Emphysema           : 0.509
Fibrosis            : 0.509
Pleural_Thickening  : 0.496
Hernia              : 0.503
Retrieved case index: 0


In [4]:
# =========================
# Explanation (NON-DECISIONAL)
# =========================
def explain(image, labels, probs, top_k=3):
    top = sorted(
        zip(labels, probs),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    summary = "\n".join([f"- {l}: {p:.2f}" for l, p in top])

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": f"""
The AI system produced the following risk suggestions:
{summary}

Explain radiological patterns that may be consistent with these findings.
Do NOT provide a diagnosis.
"""
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200
        )

    return processor.decode(out[0], skip_special_tokens=True)

print(explain(img, LABELS, probs))

user




The AI system produced the following risk suggestions:
- Fibrosis: 0.51
- Emphysema: 0.51
- Mass: 0.51

Explain radiological patterns that may be consistent with these findings.
Do NOT provide a diagnosis.
model

